In [1]:
#import libraries for calculating the traning time 
import os
import time
from pathlib import Path

#import deep learning Architecture libraries 
import torch                                         #handles multi-dimensional arrays (tensors) and automatic differentiation math.
import torch.nn as nn                                # (Neural Networks): Standard network components like layers, weights, and mathematical activation functions.
import torch.optim as optim                          #Optimization engines that adjust the model's brain based on its training errors.
from torch.utils.data import DataLoader              #preparing data for maximum utilixation of GPU
from torchvision import datasets, transforms, models # provides pre-built functions for loading, manipulating, and feeding images or videos directly into neural networks.

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.12.0+cu130


In [2]:
# Check for graphics card hardware acceleration
if torch.cuda.is_available():
    device = torch.device("cuda") #This creates an environment object
    print(f"[+] Found NVIDIA GPU: {torch.cuda.get_device_name(0)}")
    print(f"[+] Total Video Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("[-] No GPU found. Falling back to CPU.")

# Core Hyperparameters for Dell 7670 
BATCH_SIZE = 16         # Commands the DataLoader to group images into bundles of 64
EPOCHS = 10            # The model will review the collection of the 12000 images 10 times to refine its accuracy.
LEARNING_RATE = 0.000022   # Controls how drastically the model alters its weights when it makes a mistake
IMAGE_SIZE = 512        # image resolution which fit the GPU size with the upper sittings 

print(f"\n[+] Execution target set to: {device.type.upper()}")
print(f"[+] Processing images at resolution: {IMAGE_SIZE}x{IMAGE_SIZE}")

[+] Found NVIDIA GPU: NVIDIA RTX A2000 8GB Laptop GPU
[+] Total Video Memory: 8.22 GB

[+] Execution target set to: CUDA
[+] Processing images at resolution: 512x512


In [3]:
# Updated Cell 3: The Augmented Dual-Pipeline Preprocessing Blueprint

# 1. Training Pipeline: Includes randomized variations to distort backgrounds
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),       # Mirrors the image 50% of the time
    transforms.RandomRotation(degrees=15),         # Tilts the dog slightly
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Adjusts lighting conditions
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 2. Validation Pipeline: Strict, clean sizing with NO distortions for pure grading
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

print("[+] Dual-pipeline preprocessing system initialized successfully.")

[+] Dual-pipeline preprocessing system initialized successfully.


In [4]:
# Updated Cell 4: Pointing to the New SSD Train Directory

# 1. Define the explicit local paths
TRAIN_DATA_DIR = "./data/train"

# 2. Create two parallel pointers to the local training split using separate transforms
base_train_dataset = datasets.ImageFolder(root=TRAIN_DATA_DIR, transform=train_transforms)
base_val_dataset = datasets.ImageFolder(root=TRAIN_DATA_DIR, transform=val_transforms)

# 3. Calculate your split allocations based on the local 12,000 images (80% / 20%)
train_size = int(0.8 * len(base_train_dataset))
val_size = len(base_train_dataset) - train_size

# 4. Slice out the training subset from the augmented pipeline
train_dataset, _ = torch.utils.data.random_split(
    base_train_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# 5. Slice out the validation subset from the clean pipeline
_, val_dataset = torch.utils.data.random_split(
    base_val_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

num_classes = len(base_train_dataset.classes)
breed_names = base_train_dataset.classes

print("[+] Local Datasets successfully decoupled from SSD!")
print(f"[+] Training dataset allocated with dynamic data augmentation: {len(train_dataset)} images")
print(f"[+] Validation dataset allocated with clean static validation: {len(val_dataset)} images")

[+] Local Datasets successfully decoupled from SSD!
[+] Training dataset allocated with dynamic data augmentation: 9600 images
[+] Validation dataset allocated with clean static validation: 2400 images


In [5]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=4,pin_memory=True) #Shuffle traindata , num workers = CPU threades used for data pathes , pin memory = using part of the RAM

val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=4,pin_memory=True) #same sittings of train data but for Validation data , shuffle is False 

print(f"[+] DataLoaders fully operational!")
print(f"[+] Training batches per epoch: {len(train_loader)} (64 images each)")
print(f"[+] Validation batches per epoch: {len(val_loader)} (64 images each)")

[+] DataLoaders fully operational!
[+] Training batches per epoch: 600 (64 images each)
[+] Validation batches per epoch: 150 (64 images each)


In [6]:
# 1. Load the modern, pre-trained EfficientNet_V2_S weights
efficientnet_weights = models.EfficientNet_V2_S_Weights.DEFAULT
model = models.efficientnet_v2_s(weights=efficientnet_weights)

# 2. Extract the incoming feature node count from the model's bottleneck
# In EfficientNet_V2_S, this connection bridge size is exactly 1280 nodes
in_features = model.classifier[1].in_features 

# 3. Mutate the final layer index [1] to output your 120 custom dog breeds
model.classifier[1] = nn.Linear(in_features, num_classes) 

# 4. Transfer the freshly mutated model straight into your GPU VRAM
model = model.to(device) 

print(f"[+] EfficientNet_V2_S Architecture loaded cleanly!")
print(f"[+] Base features verified. Connection bridge size: {in_features} nodes")
print(f"[+] Final Layer mutated successfully to output exactly {num_classes} classes.")
print(f"[+] Neural network successfully loaded into: {next(model.parameters()).device}")

[+] EfficientNet_V2_S Architecture loaded cleanly!
[+] Base features verified. Connection bridge size: 1280 nodes
[+] Final Layer mutated successfully to output exactly 120 classes.
[+] Neural network successfully loaded into: cuda:0


In [7]:
# 1. Define the Loss Function (The Grading System)
criterion = nn.CrossEntropyLoss() #Loss Function

# 2. Define the Optimizer (The Correction Mechanism)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) # Optimizer

# 3. Initialize the Gradient Scaler (The 16-bit Hardware Accelerator)
scaler = torch.amp.GradScaler('cuda') #Hardware manager

print("[+] Optimization and Grading systems successfully initialized.")
print(f"[+] Optimizer tracking weights with step size (Learning Rate): {LEARNING_RATE}")
print("[+] Hardware Gradient Scaler armed for 16-bit floating point math.")

[+] Optimization and Grading systems successfully initialized.
[+] Optimizer tracking weights with step size (Learning Rate): 2.2e-05
[+] Hardware Gradient Scaler armed for 16-bit floating point math.


In [8]:
def train_one_epoch(epoch_index,model,train_loader,val_loader,criterion,optimizer,scaler,device) :  #define train epoch arguments
    start_time = time.time() # records the start time to calculate how long time consumed in traning 
    
    
    #************************************************  PART ONE : TRANING  *************************************************
    model.train()  #prepare model for training (Enable traning)
    running_loss = 0.0 
    correct_train = 0 #track of the exact number of individual dog images that the model correctly classifies
    total_train = 0   #track of the total number of images processed
    for batch_idx, (images, labels) in enumerate(train_loader): #train_loader delivers 64 images and their matching 64 numeric breed labels. The enumerate function assigns a step counter (batch_idx) starting at 0 and ending at 149.
        images = images.to(device, non_blocking=True) #Transfers images from Ram to GPU , non blocking tells the CPU not to stop
        labels = labels.to(device, non_blocking=True) #Transfers labels from Ram to GPU , nin_blocking tells CPU not to stop
        optimizer.zero_grad(set_to_none=True) #Clears the old Gradient error values which were accumilated during previous Loops
        
        #The Forward Pass & Grading
        with torch.amp.autocast('cuda'): #forces GPU's Tensor cores to use 16 bit floating numbers instead of 32
             outputs = model(images) #Streams the 64 images through all 50 layers of the network. The model outputs a matrix named outputs containing 120 prediction scores for every single image in the batch.
             loss = criterion(outputs, labels) # Computes the mathematical penalty for incorrect guesses.
        #The Backward Pass & Learning
        scaler.scale(loss).backward() # scaler multiplies our small 16-bit loss value by a large factor to keep the math numerically stable. Then, .backward() triggers calculus chains to compute the error gradients for all 23 million weights inside the model.
        scaler.step(optimizer) # Adam optimizer reviews the freshly calculated error gradients and tweaks your model's 23 million weights by a tiny fraction ($0.001$), making them slightly more accurate.
        scaler.update() #Examines if any calculations overflowed or broke during this step, adjusting the multiplier for the upcoming batch.
        # Track statistics for monitoring performance(Metric Accumulation)
        running_loss += loss.item() * images.size(0) #Extracts the numeric loss value and adds it to our running counter.
        _, predicted = outputs.max(1) #Extracts the highest breed guess for each image.
        total_train += labels.size(0) #Increases our total image counter by the number of items in this batch (64).
        correct_train += predicted.eq(labels).sum().item() # Computes how many images were guessed correctly in this batch.
    #Training Summary Calculation
    epoch_loss = running_loss / len(train_loader.dataset) #Divides the total accumulated training loss by all 9,600 images to find the true average error for this epoch.
    epoch_acc = (correct_train / total_train) * 100 #Divides correct guesses by total images and multiplies by 100 to yield a clean final training accuracy percentage
    
    
    #*************************************************  PART TWO : VALIDATION  ************************************************************
    model.eval() #switches model to evaluation mode (Enable evaluation)
    val_loss = 0.0 #Creates fresh tracking variables set to zero
    correct_val = 0 #Creates fresh tracking variables set to zero
    total_val = 0 #Creates fresh tracking variables set to zero

    # Stops the gradient engine completely to save memory and processing time
    with torch.no_grad(): #Tells PyTorch: "Do not track calculation histories or compute any gradients for the code inside this block." This saves massive amounts of VRAM and processing power
        #The Validation Batch Loop
        for images, labels in val_loader: # Begins a loop that runs 38 times, fetching 64 validation test images at a time from your val_loader.
            images = images.to(device, non_blocking=True) #Transfers the test images to the GPU memory slots using the exact same hardware mechanisms as the training phase.
            labels = labels.to(device, non_blocking=True) #Transfers the test answers to the GPU memory slots using the exact same hardware mechanisms as the training phase.


            with torch.amp.autocast('cuda'): #Executes a Forward Pass Only. The model makes guesses, and the criterion calculates the error penalty (No Optimization)
                outputs = model(images)
                loss = criterion(outputs, labels)

            #Accumulates testing scores, extracts predictions, tracks accuracy counts, and calculates correctness values using identical math to the training section.
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    # Calculate final validation metrics for this pass (Final Reporting)
    epoch_val_loss = val_loss / len(val_loader.dataset) #Computes the true average error score across all 2,400 hidden test images.
    epoch_val_acc = (correct_val / total_val) * 100    #Calculates  the model's real-world accuracy percentage on images it has never seen before.
    elapsed = time.time() - start_time   #Captures the current time, subtracts our initial start_time stamp, and outputs exactly how many seconds elapsed during this complete loop pass.


    # Print out a clear performance report card for this epoch
    print(f"Epoch [{epoch_index+1}] completed in {elapsed:.1f}s -> "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% || "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")      

In [9]:
print("="*50)
print("  LAUNCHING COGNITIVE ENGINE: STARTING MODEL TRAINING  ")
print("="*50 + "\n")

for epoch in range(EPOCHS): #This loop counts from 0 to 9 (matching our hyperparameter EPOCHS = 10)
    train_one_epoch(
        epoch_index=epoch,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device
    )

print("\n" + "="*50)
print("[+] TRAINING COMPLETION SIGNAL DETECTED.")
print("="*50)

# 2. Serialize and save the final trained model weights to your SSD
output_filename = "stanford_dogs_resnet50.pth"
torch.save(model.state_dict(), output_filename)

print(f"[+] Model brain saved cleanly as: '{output_filename}'")
print("[+] System offline. Ready for inference deployment!")

  LAUNCHING COGNITIVE ENGINE: STARTING MODEL TRAINING  

Epoch [1] completed in 252.2s -> Train Loss: 3.5830 | Train Acc: 45.05% || Val Loss: 1.5813 | Val Acc: 84.08%
Epoch [2] completed in 246.3s -> Train Loss: 1.4060 | Train Acc: 80.93% || Val Loss: 0.5781 | Val Acc: 88.25%
Epoch [3] completed in 247.3s -> Train Loss: 0.7166 | Train Acc: 87.36% || Val Loss: 0.3712 | Val Acc: 90.58%
Epoch [4] completed in 251.0s -> Train Loss: 0.4708 | Train Acc: 90.58% || Val Loss: 0.3278 | Val Acc: 90.79%
Epoch [5] completed in 248.4s -> Train Loss: 0.3472 | Train Acc: 92.46% || Val Loss: 0.3447 | Val Acc: 89.58%
Epoch [6] completed in 248.6s -> Train Loss: 0.2723 | Train Acc: 93.62% || Val Loss: 0.3161 | Val Acc: 90.83%
Epoch [7] completed in 249.4s -> Train Loss: 0.2164 | Train Acc: 94.84% || Val Loss: 0.3529 | Val Acc: 89.67%
Epoch [8] completed in 249.8s -> Train Loss: 0.1870 | Train Acc: 95.35% || Val Loss: 0.3339 | Val Acc: 90.25%
Epoch [9] completed in 251.3s -> Train Loss: 0.1696 | Train Acc

In [10]:
# Updated Cell 10: 100% Offline Unbiased Testing for EfficientNet_V2_S

print("[*] Loading local independent Test Split from SSD...")

TEST_DATA_DIR = "./data/test" 
local_test_dataset = datasets.ImageFolder(root=TEST_DATA_DIR, transform=val_transforms)

# Sync label dictionaries
local_test_dataset.class_to_idx = base_train_dataset.class_to_idx

local_test_loader = DataLoader(
    local_test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

# Rebuild the pristine evaluation architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_brain = models.efficientnet_v2_s()
test_brain.classifier[1] = nn.Linear(test_brain.classifier[1].in_features, num_classes)

# Save current weights first then load them to verify disk serialization
torch.save(model.state_dict(), "stanford_dogs_efficientnet.pth")
test_brain.load_state_dict(torch.load("stanford_dogs_efficientnet.pth", map_location=device))
test_brain = test_brain.to(device)
test_brain.eval()

local_test_correct = 0
local_test_total = 0

print(f"[+] Streaming {len(local_test_dataset)} fresh test images from SSD to GPU...")

with torch.no_grad():
    for images, labels in local_test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            outputs = test_brain(images)
            
        _, predicted = outputs.max(1)
        local_test_total += labels.size(0)
        local_test_correct += predicted.eq(labels).sum().item()

final_unbiased_grade = (local_test_correct / local_test_total) * 100

print("\n" + "="*50)
print(f"  EFFICIENTNET_V2_S TRUE UNBIASED LOCAL TEST ACCURACY: {final_unbiased_grade:.2f}%  ")
print("="*50)

[*] Loading local independent Test Split from SSD...
[+] Streaming 8580 fresh test images from SSD to GPU...

  EFFICIENTNET_V2_S TRUE UNBIASED LOCAL TEST ACCURACY: 91.76%  
